# Research Training Runbook — Skin Lesion Triage

> **STOP: this notebook is not ready to run for dissertation metrics.** Do not use “Run all.” Dataset license/terms and institutional approvals need supervisor confirmation, and the clinical interpretation of `akiec` in the existing binary mapping is a pending research decision. No model results are provided here; completing this runbook does not establish clinical readiness.

This is a gated, ordered runbook, not an automatic training pipeline. Record research decisions outside this notebook. Human approvals cannot be inferred or granted by code. The split is by HAM10000 `lesion_id`; no verified patient identifier is available here, so do not describe it as patient-level separation.

## Ordered gates
1. Confirm supervisor/research approval for dataset terms, required institutional approvals, and the `akiec` interpretation. Stop if any is unresolved.
2. Stage approved project/data files, install runtime dependencies, and run the approval gate below.
3. Prepare the dataset and manually review its lesion-disjoint manifest and class distributions. Stop on any unexpected/missing group, class, count, or proportion.
4. Only after documenting data review, train candidate architectures and compare them using validation results only.
5. Select one architecture and lock its validation threshold. Then explicitly run the held-out test evaluation once. Never feed test results back into model/threshold selection.

## 1. Upload the project archive and select a GPU runtime (optional)

In [ ]:
from google.colab import files
print('Select the reviewed project zip. Do not include Kaggle credentials in the archive.')
uploaded = files.upload()

In [ ]:
import zipfile
from pathlib import Path

zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_files) != 1:
    raise ValueError(f'Upload exactly one project zip; found {len(zip_files)}.')
with zipfile.ZipFile(zip_files[0]) as archive:
    archive.extractall('/content/')
%cd /content/skin_lesion_triage
!ls

## 2. Install the project-pinned dependencies, record runtime, and set seeds

This notebook accepts Python 3.10–3.12 only, matching the project's supported runtime. The setup cell stops before installation on other versions, installs `requirements.txt`, and the next cell prints the actual versions and available GPU. The fixed seeds and TensorFlow deterministic-op setting improve repeatability, but do not guarantee bit-identical runs across hardware/software versions.

In [ ]:
import sys
import subprocess

if not ((3, 10) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        f'Unsupported Python {sys.version_info.major}.{sys.version_info.minor}; '
        'use Python 3.10, 3.11, or 3.12 for this project.'
    )
print(f'Python {sys.version.split()[0]} is within the supported range.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

In [ ]:
import os
import random
import platform
from importlib.metadata import version

SEED = 42
os.environ['TF_DETERMINISTIC_OPS'] = '1'
import cv2
import numpy as np
import sklearn
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('Keras:', version('keras'))
print('NumPy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('OpenCV:', cv2.__version__)
print('Seed (Python/NumPy/TensorFlow):', SEED)
print('TensorFlow deterministic ops: enabled')
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', [device.name for device in gpus] if gpus else 'none (CPU runtime)')

def run_seeded_script(script, *args):
    # The project scripts run in child processes, so seed each child before
    # importing/running the requested script rather than relying on kernel RNG state.
    seed_runner = (
        'import random, runpy, sys; import numpy as np; import tensorflow as tf; '
        f'random.seed({SEED}); np.random.seed({SEED}); '
        f'tf.keras.utils.set_random_seed({SEED}); '
        'tf.config.experimental.enable_op_determinism(); '
        'script, *args = sys.argv[1:]; sys.argv = [script, *args]; '
        "runpy.run_path(script, run_name='__main__')"
    )
    subprocess.run([sys.executable, '-c', seed_runner, script, *args], check=True)

In [ ]:
# Change these only after documenting the corresponding decisions/approvals.
SUPERVISOR_APPROVED_DATA_TERMS = False
INSTITUTIONAL_APPROVAL_CONFIRMED = False
AKIEC_INTERPRETATION_RESOLVED = False

if not all((SUPERVISOR_APPROVED_DATA_TERMS, INSTITUTIONAL_APPROVAL_CONFIRMED, AKIEC_INTERPRETATION_RESOLVED)):
    raise RuntimeError('STOP: obtain/document every approval and resolve akiec interpretation before using HAM10000.')
print('Human approval gate acknowledged. Continue only within the approved scope.')

## 3. Stage and prepare approved HAM10000 data

Prefer staging the approved extracted dataset under `data/raw/` and selecting `staged`. The optional `kaggle` mode requires the Kaggle CLI and credentials configured independently; never upload/commit credentials. Do not use the download path until the approvals above are complete.

In [ ]:
DATA_SOURCE = 'staged'  # 'kaggle' is available only after the approval gate.
if DATA_SOURCE == 'staged':
    command = ('scripts/prepare_data.py', '--skip_download')
elif DATA_SOURCE == 'kaggle':
    command = ('scripts/prepare_data.py', '--kaggle_download')
else:
    raise ValueError(f'Unsupported DATA_SOURCE: {DATA_SOURCE}')
run_seeded_script(command[0], *command[1:], '--seed', str(SEED))

In [ ]:
import json

manifest_path = Path('data/processed/split_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['grouping_column'] == 'lesion_id'
assert manifest['group_disjoint'] is True
assert all(not overlap for overlap in manifest['group_overlaps'].values())
print('Seed:', manifest['seed'])
for split, details in manifest['splits'].items():
    print(split, details)
print('STOP and manually review the manifest, image/class counts, source terms, and class proportions before the next gate.')

In [ ]:
DATA_PREP_REVIEW_CONFIRMED = False
if not DATA_PREP_REVIEW_CONFIRMED:
    raise RuntimeError('STOP: document and confirm manual manifest/data review before training.')

## 4. Train architecture candidates (validation only)

Training uses only train/validation directories. Dataset images pass through the same deterministic OpenCV hair-removal, denoising, CLAHE, area-resize, and RGB conversion functions used by serving. The blur/quality rejection remains inference-only so it does not selectively remove training cases using an unvalidated threshold; validate that intake gate separately on representative intended-use images. The shared image transform establishes implementation parity, not evidence that these operations improve generalization.

Architecture and threshold decisions must be based only on validation evidence. Run candidates separately as needed; neither command evaluates the held-out test set.

In [ ]:
run_seeded_script('scripts/train_model.py', '--arch', 'mobilenetv2', '--epochs_head', '10', '--epochs_finetune', '15')

In [ ]:
run_seeded_script('scripts/train_model.py', '--arch', 'resnet50', '--epochs_head', '10', '--epochs_finetune', '15')

## 5. Lock a model and make the one-time final test decision

Review validation results and thresholds. Select exactly one architecture before looking at test metrics. Record the decision and do not tune/retrain based on the final test result. The code separates this command from training, but cannot prevent a person from running it repeatedly.

In [ ]:
SELECTED_ARCH = None  # Set to 'mobilenetv2' or 'resnet50' after validation-only selection.
FINAL_TEST_EVALUATION_AUTHORIZED = False  # Set True only after locking architecture + threshold.
if SELECTED_ARCH not in ('mobilenetv2', 'resnet50'):
    raise ValueError('STOP: choose one architecture using validation evidence only.')
if not FINAL_TEST_EVALUATION_AUTHORIZED:
    raise RuntimeError('STOP: lock the validation threshold and authorize the one-time final test evaluation.')

In [ ]:
run_seeded_script('scripts/train_model.py', '--arch', SELECTED_ARCH, '--final-evaluate')

## 6. Export the trained models and their locked artifacts

Each architecture run writes three files into `models/`: the `.keras` weights, the matching `<arch>_threshold.json`, and, only after the authorized final evaluation, `<arch>_final_test_metrics.json`. All three are required to reproduce or report a result, so export them together rather than the `.keras` file alone.

In [ ]:
from google.colab import files
import os

export_dir = 'models'
exportable = [
    f for f in os.listdir(export_dir)
    if f.endswith('.keras') or f.endswith('_threshold.json') or f.endswith('_final_test_metrics.json')
]
if not exportable:
    print(f"No exportable files found in '{export_dir}'. Train and/or run the final evaluation first.")
else:
    for fname in sorted(exportable):
        print('Downloading', fname)
        files.download(os.path.join(export_dir, fname))

## 7. (Optional) Save the same files straight to Google Drive instead

Useful if the direct download above times out for large files, or to keep a durable copy outside the ephemeral Colab runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

export_dir = 'models'
dest = '/content/drive/MyDrive/skin_lesion_triage_models'
os.makedirs(dest, exist_ok=True)

exportable = [
    f for f in os.listdir(export_dir)
    if f.endswith('.keras') or f.endswith('_threshold.json') or f.endswith('_final_test_metrics.json')
]
if not exportable:
    print(f"No exportable files found in '{export_dir}'. Train and/or run the final evaluation first.")
else:
    for fname in sorted(exportable):
        shutil.copy(os.path.join(export_dir, fname), dest)
        print('Copied', fname, 'to Drive:', dest)

## Interpretation boundary

The final test metrics are descriptive results for this dataset/split and locked decision threshold. Do not retune from them or present them as clinical validation. The existing `akiec` mapping and dataset/institutional approvals remain research governance matters; preprocessing and split checks cannot resolve them. No model artifact produced here is clinically ready by virtue of training or a passing metric.